# Phillips Curve Across U.S. States
## Texas, Massachusetts, and Ohio  |  2000 – Present

---

### What is the Phillips Curve?

The **Phillips Curve** is one of the most influential ideas in macroeconomics. First described by economist A. W. Phillips in 1958, it posits an inverse relationship between the **rate of unemployment** and the **rate of inflation**: when unemployment is low, employers compete for scarce workers and bid wages up, which pushes prices higher; conversely, when unemployment is high, wage and price pressures ease and inflation tends to fall. Although the simple short-run trade-off has been complicated by stagflation, supply shocks, inflation expectations, and the introduction of the Non-Accelerating Inflation Rate of Unemployment (NAIRU), the Phillips Curve remains a central framework for central banks, labor economists, and policy analysts seeking to understand the dynamics of prices and the labor market.

### About This Notebook

This notebook — `01_data_collection.ipynb` — is the **first step** in the project. Its sole job is **data collection**: connecting to the Federal Reserve Economic Data (FRED) API, pulling state-level unemployment and inflation series for Texas, Massachusetts, and Ohio from 2000 onward, and persisting the raw data locally for use in subsequent cleaning, analysis, and modeling notebooks.

In [ ]:
# ---------------------------------------------------------------
# macOS SSL certificate fix.
# Python on macOS sometimes can't find the system trust store, which
# causes "certificate verify failed" errors when fredapi makes HTTPS
# calls. Pointing the SSL stack at certifi's bundled CA roots fixes
# this. Must run BEFORE any API calls (so before fredapi is used).
# ---------------------------------------------------------------
import os
import ssl
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

# ---------- Standard project imports ----------
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from fredapi import Fred


# ---------------------------------------------------------------
# FRED client factory with rate-limit retry.
# FRED limits ~120 requests / 60 seconds. Re-running cells or pulling
# many series in a tight loop can trip "Too Many Requests" errors.
# This factory builds a Fred client and wraps its get_series method
# so any rate-limit ValueError is caught and retried with exponential
# backoff (30s, 60s, 120s, 240s, 480s). Subsequent data-pull cells
# don't need to change -- they just call fred.get_series() as before.
# ---------------------------------------------------------------
def make_fred_with_retry(api_key, max_retries=5):
    client = Fred(api_key=api_key)
    _original_get_series = client.get_series

    def _get_series_with_retry(series_id, **kwargs):
        for attempt in range(max_retries):
            try:
                return _original_get_series(series_id, **kwargs)
            except ValueError as e:
                msg = str(e)
                # fredapi surfaces FRED's rate-limit response as a ValueError
                # with these strings in the message.
                if 'Rate Limit' in msg or 'Too Many Requests' in msg:
                    wait = 30 * (2 ** attempt)  # 30, 60, 120, 240, 480
                    print(
                        f"  Rate-limited on {series_id}; sleeping {wait}s "
                        f"(attempt {attempt + 1}/{max_retries})..."
                    )
                    time.sleep(wait)
                else:
                    # Any other ValueError (e.g. bad series ID) propagates.
                    raise
        raise RuntimeError(
            f"FRED rate limit exceeded for {series_id} after {max_retries} retries."
        )

    # Replace the bound method so all later code that calls
    # fred.get_series() automatically benefits from retries.
    client.get_series = _get_series_with_retry
    return client


print("All libraries loaded successfully.")

In [ ]:
# Connect to the FRED API
# The API key is read from ../fred_api_key.txt (project root) so it is never hardcoded.

key_path = os.path.join('..', 'fred_api_key.txt')

try:
    with open(key_path, 'r') as f:
        fred_api_key = f.read().strip()
except FileNotFoundError:
    print(
        f"ERROR: Could not find the API key file at '{key_path}'.\n"
        "Please create a file named 'fred_api_key.txt' in the project root "
        "(one level above the notebooks/ folder) and paste your FRED API key into it."
    )
    raise

# Use the factory from cell 1 so the returned client automatically
# retries on FRED rate-limit errors.
fred = make_fred_with_retry(fred_api_key)

# Test the connection by pulling one data point from UNRATE (U.S. civilian unemployment rate).
# If FRED rate-limits this call, the retry wrapper will sleep and try again automatically.
test_series = fred.get_series('UNRATE').tail(1)
print("FRED connection successful.")
print(f"Latest UNRATE observation: {test_series.index[0].date()} -> {test_series.iloc[0]}%")

## State-Level Unemployment Data

The state-level unemployment data used in this project comes from the **U.S. Bureau of Labor Statistics (BLS) Local Area Unemployment Statistics (LAUS)** program, accessed through the FRED API. LAUS produces monthly estimates of total employment and unemployment for states (and sub-state areas) using a model-based approach that combines the Current Population Survey, the Current Employment Statistics survey, and state unemployment insurance claims. The series pulled here — `TXUR` (Texas), `MAUR` (Massachusetts), and `OHUR` (Ohio) — are **monthly, seasonally adjusted** unemployment rates expressed as a percentage of the state labor force.

In [ ]:
# ---------------------------------------------------------------
# Pull monthly, seasonally adjusted unemployment rates for the
# three states from FRED, starting in January 2000.
# ---------------------------------------------------------------

# Safety net: if this cell is run in a fresh kernel without first
# executing the earlier setup cells, `fred` (the FRED API client)
# will not exist. Re-create it here -- using the retry-enabled
# factory if it's available, otherwise a plain Fred client.
if 'fred' not in dir():
    import os
    with open(os.path.join('..', 'fred_api_key.txt'), 'r') as _f:
        _key = _f.read().strip()
    if 'make_fred_with_retry' in dir():
        fred = make_fred_with_retry(_key)
    else:
        from fredapi import Fred
        fred = Fred(api_key=_key)

# Map FRED series IDs to human-readable state names so we can label rows later.
state_series = {
    'TXUR': 'Texas',
    'MAUR': 'Massachusetts',
    'OHUR': 'Ohio',
}

# Define the observation window. observation_start is passed to FRED;
# leaving observation_end unset pulls through the most recent release.
start_date = '2000-01-01'

# Collect one tidy DataFrame per state, then concatenate.
frames = []
for series_id, state_name in state_series.items():
    # fred.get_series returns a pandas Series indexed by date.
    series = fred.get_series(series_id, observation_start=start_date)

    # Convert the Series into a long-format DataFrame:
    #   - 'date' column from the Series index
    #   - 'state' column with the state name (constant for this slice)
    #   - 'unemployment_rate' column with the actual values
    df_state = pd.DataFrame({
        'date': series.index,
        'state': state_name,
        'unemployment_rate': series.values,
    })
    frames.append(df_state)

# Stack the three per-state DataFrames vertically into one long-format table.
# ignore_index=True resets the row index so it runs 0..N-1 across all states.
unemployment_df = pd.concat(frames, ignore_index=True)

# Force the date column to pandas datetime dtype (it usually already is,
# but being explicit guards against future API changes).
unemployment_df['date'] = pd.to_datetime(unemployment_df['date'])

# Sort by state then date so the table reads naturally when inspected.
unemployment_df = unemployment_df.sort_values(['state', 'date']).reset_index(drop=True)

# ---------------------------------------------------------------
# Inspect the result
# ---------------------------------------------------------------

# Shape: (rows, columns). Expect ~ (months_since_2000 * 3, 3).
print(f"DataFrame shape: {unemployment_df.shape}")
print()

# First 10 rows give a quick visual sanity check of the long format.
print("First 10 rows:")
print(unemployment_df.head(10))
print()

# Min/max of the date column show the observation window actually returned.
print(f"Date range: {unemployment_df['date'].min().date()} to {unemployment_df['date'].max().date()}")
print()

# groupby('state') splits the long table by state; .agg() computes
# mean / min / max for the unemployment_rate column within each group.
print("Summary statistics by state (unemployment_rate, %):")
summary = unemployment_df.groupby('state')['unemployment_rate'].agg(['mean', 'min', 'max']).round(2)
print(summary)

In [ ]:
# ---------------------------------------------------------------
# Line chart: unemployment rate over time for all three states
# ---------------------------------------------------------------

# Make sure the figures/ output directory exists before saving anything.
# exist_ok=True prevents an error if the folder was already created.
figures_dir = os.path.join('..', 'figures')
os.makedirs(figures_dir, exist_ok=True)

# Use seaborn's default style for cleaner gridlines and typography.
sns.set_style('whitegrid')

# Create a single Axes object; figsize is in inches (width, height).
fig, ax = plt.subplots(figsize=(12, 6))

# Loop through each state, filter the long DataFrame, and plot one line per state.
# Plotting in a loop is cleaner than pivoting when there are only a few series.
for state_name in state_series.values():
    subset = unemployment_df[unemployment_df['state'] == state_name]
    ax.plot(subset['date'], subset['unemployment_rate'], label=state_name, linewidth=1.5)

# Title and axis labels.
ax.set_title('State Unemployment Rates (Seasonally Adjusted), 2000–Present', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Unemployment Rate (%)')

# Legend draws from the label= argument supplied in each ax.plot() call.
ax.legend(title='State', loc='upper right')

# tight_layout trims excess padding around the figure.
plt.tight_layout()

# Save the figure before showing it. dpi=150 keeps text crisp at this size.
output_path = os.path.join(figures_dir, 'unemployment_raw.png')
plt.savefig(output_path, dpi=150)
print(f"Saved figure to: {output_path}")

# Render the chart inline in the notebook.
plt.show()

## State-Level Inflation Data

The U.S. Bureau of Labor Statistics (BLS) does **not** publish the Consumer Price Index (CPI) for every state directly. To approximate state-level inflation, this project combines two BLS sources:

1. **Primary — metropolitan-area CPI.** The CPI for the largest metro area in each state is used as a proxy for state-level prices: Houston-The Woodlands-Sugar Land (TX), Boston-Cambridge-Newton (MA), and Cleveland-Akron (with Cincinnati as a backup) for OH.

2. **Fill / fallback — BLS regional CPI.** Some metro series are released only bimonthly (so they have gaps in alternating months) and others were discontinued by BLS (Cleveland-Akron, for example, stops in 2017–2018). To produce a complete monthly series for every state, the code below **splices in BLS Census-region CPI** — South (TX), Northeast (MA), and Midwest (OH) — for any month where the metro series is unavailable. The metro source is preferred whenever it exists; the regional source fills the rest.

The result is a fully monthly inflation series for each state running from 2000 to the most recent release. The cell prints how many observations came from the metro source versus the regional fill so the split is transparent. YoY inflation is computed for each source independently (using a calendar-month shift of 12 months) and then spliced — this avoids any artificial jump that could be introduced by mixing two index series with different base levels.

In [ ]:
# ---------------------------------------------------------------
# Pull metro CPI + regional CPI, splice them to fill gaps,
# and compute year-over-year inflation per state.
# ---------------------------------------------------------------

# Safety net: re-create FRED client if this cell is run in isolation.
if 'fred' not in dir():
    import os
    from fredapi import Fred
    with open(os.path.join('..', 'fred_api_key.txt'), 'r') as _f:
        fred = Fred(api_key=_f.read().strip())

# Candidate metro-area CPI series IDs to try, in order of preference,
# for each state. The first candidate that returns >= MIN_OBS observations
# is used as the metro source.
metro_candidates = {
    'TX': [
        ('CUURA318SA0', 'Houston-The Woodlands-Sugar Land, TX (CPI-U)'),
        ('CUURA316SA0', 'Dallas-Fort Worth-Arlington, TX (CPI-U)'),
    ],
    'MA': [
        ('CUURA103SA0', 'Boston-Cambridge-Newton, MA-NH (CPI-U)'),
    ],
    'OH': [
        ('CUURA210SA0', 'Cleveland-Akron, OH (CPI-U)'),
        ('CUURA213SA0', 'Cincinnati-Hamilton, OH-KY-IN (CPI-U)'),
    ],
}

# Regional CPI series IDs (monthly, full history back to 2000+).
# Used to fill any month where the metro series has no observation.
regional_fallback = {
    'TX': ('CUUR0300SA0', 'South Census Region (CPI-U)'),
    'MA': ('CUUR0100SA0', 'Northeast Census Region (CPI-U)'),
    'OH': ('CUUR0200SA0', 'Midwest Census Region (CPI-U)'),
}

start_date = '2000-01-01'
MIN_OBS = 100  # minimum number of metro observations to use metro at all

cpi_frames = []   # per-state DataFrames, concatenated at the end
source_log = {}   # state -> dict with bookkeeping for the print summary

for state in ['TX', 'MA', 'OH']:
    # ---------- 1. Try to obtain a usable metro CPI series ----------
    metro = None
    metro_id = None
    metro_desc = None
    for series_id, desc in metro_candidates[state]:
        try:
            s = fred.get_series(series_id, observation_start=start_date).dropna()
            if len(s) >= MIN_OBS:
                metro, metro_id, metro_desc = s, series_id, desc
                break
            else:
                print(f"  {state}: metro {series_id} has only {len(s)} obs (< {MIN_OBS}); trying next.")
        except Exception as e:
            print(f"  {state}: metro {series_id} unavailable ({type(e).__name__}); trying next.")

    # ---------- 2. Always pull the regional CPI series ----------
    # Regional CPI is monthly and has full coverage, so it can fill any
    # gaps left by the bimonthly / discontinued metro series.
    regional_id, regional_desc = regional_fallback[state]
    regional = fred.get_series(regional_id, observation_start=start_date).dropna()

    # ---------- 3. Build a complete monthly date index ----------
    # End at whichever source extends furthest (regional usually does).
    end_date = regional.index.max()
    if metro is not None and metro.index.max() > end_date:
        end_date = metro.index.max()
    monthly_idx = pd.date_range(start=start_date, end=end_date, freq='MS')

    # ---------- 4. Compute YoY inflation on each source INDEPENDENTLY ----------
    # Computing YoY on each native series and then splicing the *rates* avoids
    # the level-mismatch problem you would get from splicing two index series
    # that have different base values (e.g. Cleveland at ~240 vs Midwest at ~260
    # in 2018 — a direct splice would create a fake -8% spike at the join).
    #
    # shift(12, freq='MS') moves the date index forward by 12 calendar months
    # while keeping the values; aligning to the original series gives the
    # 12-months-prior value at every date — robust to bimonthly cadence.
    regional_yoy = ((regional / regional.shift(12, freq='MS')) - 1) * 100

    if metro is not None:
        metro_yoy = ((metro / metro.shift(12, freq='MS')) - 1) * 100

    # ---------- 5. Reindex both to the monthly grid, then splice ----------
    regional_m = regional.reindex(monthly_idx)
    regional_yoy_m = regional_yoy.reindex(monthly_idx)

    if metro is not None:
        metro_m = metro.reindex(monthly_idx)
        metro_yoy_m = metro_yoy.reindex(monthly_idx)

        # combine_first: take values from the caller (metro) where present,
        # fill remaining NaNs with values from the argument (regional).
        # This is the splice — metro wins wherever it exists, regional fills
        # the off-months of bimonthly cadence and the post-discontinuation tail.
        cpi_combined = metro_m.combine_first(regional_m)
        yoy_combined = metro_yoy_m.combine_first(regional_yoy_m)

        # Per-row flag so we can audit which source ended up filling each month.
        source_per_row = np.where(metro_m.notna(), 'metro', 'regional')
    else:
        # No usable metro series at all — use regional everywhere.
        cpi_combined = regional_m
        yoy_combined = regional_yoy_m
        source_per_row = np.array(['regional'] * len(monthly_idx))

    # ---------- 6. Build the per-state long-format DataFrame ----------
    df_state = pd.DataFrame({
        'date': monthly_idx,
        'state': state,
        'cpi_index': cpi_combined.values,
        'inflation_rate_yoy': yoy_combined.values,
        'cpi_source': source_per_row,
    })
    # Drop rows where the CPI index is still NaN (defensive — regional should
    # always cover the full range, but guard against any edge case).
    df_state = df_state.dropna(subset=['cpi_index']).reset_index(drop=True)
    cpi_frames.append(df_state)

    # ---------- 7. Record bookkeeping for the printed summary ----------
    n_metro = int((df_state['cpi_source'] == 'metro').sum())
    n_regional = int((df_state['cpi_source'] == 'regional').sum())
    source_log[state] = {
        'metro_id': metro_id,
        'metro_desc': metro_desc,
        'regional_id': regional_id,
        'regional_desc': regional_desc,
        'n_metro': n_metro,
        'n_regional': n_regional,
        'total': len(df_state),
    }

# Concatenate all states into one long-format DataFrame.
inflation_df = pd.concat(cpi_frames, ignore_index=True)
inflation_df = inflation_df.sort_values(['state', 'date']).reset_index(drop=True)

# ---------------------------------------------------------------
# Report which source filled each state's series
# ---------------------------------------------------------------
print("\nCPI source per state (metro CPI with regional fill where missing):")
for state, info in source_log.items():
    if info['metro_id'] is None:
        print(f"  {state}: REGIONAL ONLY  {info['regional_id']}  ({info['regional_desc']}, {info['total']} obs)")
    else:
        print(
            f"  {state}: metro {info['metro_id']} ({info['n_metro']} obs) "
            f"+ regional {info['regional_id']} fill ({info['n_regional']} obs) "
            f"= {info['total']} total"
        )
        print(f"        metro:    {info['metro_desc']}")
        print(f"        regional: {info['regional_desc']}")

# ---------------------------------------------------------------
# Inspect the combined DataFrame
# ---------------------------------------------------------------
print(f"\nDataFrame shape: {inflation_df.shape}")

print("\nFirst 10 rows:")
print(inflation_df.head(10))

print("\nSummary statistics by state (inflation_rate_yoy, %):")
infl_summary = (
    inflation_df
    .groupby('state')['inflation_rate_yoy']
    .agg(['mean', 'min', 'max', 'count'])
    .round(2)
)
print(infl_summary)

# Sanity check: overall mean YoY inflation should sit between ~1.5% and ~4%
# for 2001-onward U.S. data. If not, the YoY math or the source series is off.
overall_mean = inflation_df['inflation_rate_yoy'].mean()
print(f"\nOverall mean YoY inflation: {overall_mean:.2f}%")
if 1.5 <= overall_mean <= 4.0:
    print("Within expected range (1.5%–4%). Calculation looks correct.")
else:
    print("WARNING: outside expected range (1.5%–4%) — investigate the source series or YoY math.")

In [ ]:
# ---------------------------------------------------------------
# Line chart: year-over-year inflation rate by state
# ---------------------------------------------------------------

# Ensure the output directory exists; exist_ok=True avoids errors on re-run.
figures_dir = os.path.join('..', 'figures')
os.makedirs(figures_dir, exist_ok=True)

# Apply seaborn's whitegrid style for consistency with the unemployment chart.
sns.set_style('whitegrid')

# A single Axes is fine; one line per state on shared y-axis (% YoY).
fig, ax = plt.subplots(figsize=(12, 6))

# Loop over states in a fixed order so colors are consistent across runs.
for state in ['TX', 'MA', 'OH']:
    subset = inflation_df[inflation_df['state'] == state]
    # dropna() so the first 12 months (which have no YoY value) don't draw weird gaps.
    subset = subset.dropna(subset=['inflation_rate_yoy'])
    ax.plot(subset['date'], subset['inflation_rate_yoy'], label=state, linewidth=1.5)

# Reference line at 0% inflation makes deflationary episodes visually obvious.
ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')

ax.set_title('Year-over-Year Inflation by State (CPI-based), 2001–Present', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('YoY Inflation Rate (%)')
ax.legend(title='State', loc='upper left')

plt.tight_layout()

# Save before show() — some matplotlib backends clear the figure on display.
output_path = os.path.join(figures_dir, 'inflation_raw.png')
plt.savefig(output_path, dpi=150)
print(f"Saved figure to: {output_path}")

# We expect to see a clearly visible spike in 2021–2023 across all three states.
plt.show()

## Control Variables

A simple bivariate regression of state inflation on state unemployment is almost certainly **misspecified**: many other forces move inflation up or down independently of local labor-market slack. Without controlling for them, our estimate of the Phillips-Curve slope would absorb their effects and suffer from **omitted variable bias**. The variables collected in this section are designed to soak up the most important of those confounders so that the remaining variation in inflation can be attributed more cleanly to state unemployment.

The controls fall into two groups:

**National monthly controls** (one value per month, the same for all states):

- **National CPI inflation (YoY)** — captures common nationwide shocks to prices (energy, supply chains, the dollar) that move every state's inflation in the same direction.
- **WTI crude oil price** — a key input cost. Particularly important for Texas, where oil shocks affect both employment *and* local prices, so failing to control for it would conflate labor-market and price-input effects.
- **30-year fixed mortgage rate** — shelter is the largest single component of CPI, and mortgage rates are the dominant driver of housing costs and the housing-services component of inflation.
- **Federal Funds Rate** — captures the overall monetary policy stance. When the Fed raises rates it is explicitly trying to slow inflation, so this variable absorbs the policy-driven part of the inflation–unemployment relationship.

**State-level quarterly control:**

- **FHFA House Price Index (HPI)** by state — captures the state-specific housing market, which feeds into local inflation through rents and the imputed cost of owner-occupied housing. HPI is published quarterly, so we pull it as-is here and align it to the monthly panel in a later notebook.

In [ ]:
# ---------------------------------------------------------------
# Pull control variables from FRED and assemble them into two
# DataFrames:
#   - national_controls_df : monthly, date-indexed, one row per month
#   - hpi_df               : quarterly, long-format (date, state, hpi)
# ---------------------------------------------------------------

# Safety net: re-create FRED client if this cell is run in isolation.
if 'fred' not in dir():
    import os
    from fredapi import Fred
    with open(os.path.join('..', 'fred_api_key.txt'), 'r') as _f:
        fred = Fred(api_key=_f.read().strip())

start_date = '2000-01-01'

# ---------- 1. National CPI -> YoY inflation (monthly) ----------
# CPIAUCSL: CPI for All Urban Consumers, All Items, Seasonally Adjusted.
# Index value (not a rate), so we compute year-over-year percent change.
cpi_national = fred.get_series('CPIAUCSL', observation_start=start_date).dropna()
# shift(12, freq='MS') moves the index forward 12 calendar months so that
# dividing the original series by it produces the YoY ratio at each date.
national_inflation_yoy = ((cpi_national / cpi_national.shift(12, freq='MS')) - 1) * 100

# ---------- 2. WTI Crude Oil Price (daily -> monthly average) ----------
# DCOILWTICO is reported on business days; .resample('MS') groups by
# month-start bins and .mean() collapses the daily prices into a single
# monthly average. This smooths out within-month volatility.
oil_daily = fred.get_series('DCOILWTICO', observation_start=start_date).dropna()
oil_price_wti = oil_daily.resample('MS').mean()

# ---------- 3. 30-Year Fixed Mortgage Rate (weekly -> monthly average) ----------
# MORTGAGE30US is the Freddie Mac weekly survey rate. We collapse to a
# monthly average so it aligns with the other monthly controls.
mortgage_weekly = fred.get_series('MORTGAGE30US', observation_start=start_date).dropna()
mortgage_rate_30yr = mortgage_weekly.resample('MS').mean()

# ---------- 4. Federal Funds Rate (already monthly) ----------
# FEDFUNDS is the effective federal funds rate, published as a monthly
# average by the Federal Reserve. No resampling needed.
fed_funds_rate = fred.get_series('FEDFUNDS', observation_start=start_date).dropna()

# ---------- Combine monthly controls into one DataFrame ----------
# Passing a dict of Series to pd.DataFrame() aligns them on their indexes
# (all month-start dates here), producing one row per month with NaN
# wherever a series is missing a value.
national_controls_df = pd.DataFrame({
    'national_inflation_yoy': national_inflation_yoy,
    'oil_price_wti': oil_price_wti,
    'mortgage_rate_30yr': mortgage_rate_30yr,
    'fed_funds_rate': fed_funds_rate,
}).sort_index()
national_controls_df.index.name = 'date'

# ---------- 5. State-level FHFA House Price Index (quarterly, left as-is) ----------
# Quarterly series — frequency alignment to the monthly panel will be
# handled in a later cleaning notebook (e.g. forward-fill or interpolation).
hpi_series_ids = {
    'TX': 'TXSTHPI',
    'MA': 'MASTHPI',
    'OH': 'OHSTHPI',
}

hpi_frames = []
for state, series_id in hpi_series_ids.items():
    s = fred.get_series(series_id, observation_start=start_date).dropna()
    # Long-format DataFrame: one row per (state, date) pair.
    hpi_frames.append(pd.DataFrame({
        'date': s.index,
        'state': state,
        'hpi': s.values,
    }))

hpi_df = pd.concat(hpi_frames, ignore_index=True)
hpi_df['date'] = pd.to_datetime(hpi_df['date'])
hpi_df = hpi_df.sort_values(['state', 'date']).reset_index(drop=True)

# ---------------------------------------------------------------
# Summary statistics
# ---------------------------------------------------------------
print("National monthly control variables — summary statistics:")
print(national_controls_df.describe().round(2))
print()
print(f"national_controls_df shape: {national_controls_df.shape}")
print(
    f"Date range: {national_controls_df.index.min().date()} "
    f"to {national_controls_df.index.max().date()} (monthly)"
)
print()

print("State-level FHFA House Price Index — summary by state (quarterly):")
print(hpi_df.groupby('state')['hpi'].agg(['mean', 'min', 'max', 'count']).round(2))
print()
print(f"hpi_df shape: {hpi_df.shape}")
print(
    f"HPI date range: {hpi_df['date'].min().date()} "
    f"to {hpi_df['date'].max().date()} (quarterly)"
)

In [ ]:
# ---------------------------------------------------------------
# 2x2 grid: oil, mortgage rate, fed funds, national inflation
# ---------------------------------------------------------------

# Ensure the figures/ output directory exists.
figures_dir = os.path.join('..', 'figures')
os.makedirs(figures_dir, exist_ok=True)

sns.set_style('whitegrid')

# subplots(2, 2) returns a (2, 2) array of Axes; figsize is in inches.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Top-left: WTI crude oil price (level, $/barrel)
axes[0, 0].plot(
    national_controls_df.index, national_controls_df['oil_price_wti'],
    color='C0', linewidth=1.5,
)
axes[0, 0].set_title('WTI Crude Oil Price (Monthly Avg)')
axes[0, 0].set_ylabel('US Dollars per Barrel')

# Top-right: 30-year fixed mortgage rate (percent)
axes[0, 1].plot(
    national_controls_df.index, national_controls_df['mortgage_rate_30yr'],
    color='C1', linewidth=1.5,
)
axes[0, 1].set_title('30-Year Fixed Mortgage Rate (Monthly Avg)')
axes[0, 1].set_ylabel('Percent')

# Bottom-left: federal funds rate (percent)
axes[1, 0].plot(
    national_controls_df.index, national_controls_df['fed_funds_rate'],
    color='C2', linewidth=1.5,
)
axes[1, 0].set_title('Federal Funds Rate')
axes[1, 0].set_ylabel('Percent')

# Bottom-right: national YoY inflation (percent change), with 0% reference line
axes[1, 1].plot(
    national_controls_df.index, national_controls_df['national_inflation_yoy'],
    color='C3', linewidth=1.5,
)
# Horizontal zero line makes deflationary periods (e.g. 2009) easy to spot.
axes[1, 1].axhline(0, color='gray', linewidth=0.6, linestyle='--')
axes[1, 1].set_title('National CPI Inflation (Year-over-Year)')
axes[1, 1].set_ylabel('Percent')

# Common x-axis label on every panel.
for ax in axes.flat:
    ax.set_xlabel('Date')

# Overall title; y=1.0 keeps it from overlapping the top row.
fig.suptitle('National Control Variables, 2000–Present', fontsize=14, y=1.00)

plt.tight_layout()

# Save the figure before display.
output_path = os.path.join(figures_dir, 'controls_raw.png')
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Saved figure to: {output_path}")

plt.show()

## State Structural Characteristics

Texas, Massachusetts, and Ohio are not just three random observations — they were chosen because they sit on very different points along several structural dimensions that economic theory says should shape the Phillips Curve. The variables collected here are **time-invariant or very slow-moving** (industry mix, unionization, education, income, population), so they will not enter the regression as time-series controls. Instead, they provide the *qualitative context* needed to interpret why the estimated Phillips-Curve slope might differ across the three states.

Among the dimensions that matter for the Phillips Curve:

- **Industry mix** — manufacturing-heavy states are more exposed to global trade and the goods cycle, while service-heavy states are more sensitive to local wage dynamics.
- **Energy share** — oil-producing states experience supply shocks that move *both* sides of the curve at once, blurring the standard inflation–unemployment trade-off.
- **Unionization** — higher union density typically means stickier nominal wages and slower wage adjustment to slack.
- **Education and income** — proxies for the share of high-skill jobs; tight skilled-labor markets tend to produce steeper wage–unemployment relationships.

The DataFrame in the next cell hardcodes representative values drawn from publicly available Census Bureau and BLS sources. The numbers are approximate and used here for context and discussion, not as regression inputs.

In [ ]:
# ---------------------------------------------------------------
# Hardcoded structural characteristics for the three states.
# Values are approximate, drawn from public Census/BLS sources;
# they are used for context, not regression inputs.
# ---------------------------------------------------------------

# Build the DataFrame from a list of dicts -- one dict per state.
# This is the most readable form for hand-entered tabular data.
state_characteristics = pd.DataFrame([
    {
        'state': 'TX',
        'manufacturing_share_pct': 8.5,
        'energy_share_pct': 12.0,
        'unionization_rate_pct': 4.7,
        'median_household_income': 67500,
        'pct_bachelors_degree': 32.0,
        'population_millions': 30.0,
        'dominant_industries': 'Oil & Gas, Technology, Healthcare, Trade',
    },
    {
        'state': 'MA',
        'manufacturing_share_pct': 9.0,
        'energy_share_pct': 0.3,
        'unionization_rate_pct': 12.5,
        'median_household_income': 89645,
        'pct_bachelors_degree': 45.0,
        'population_millions': 7.0,
        'dominant_industries': 'Healthcare, Higher Education, Technology, Finance',
    },
    {
        'state': 'OH',
        'manufacturing_share_pct': 16.5,
        'energy_share_pct': 1.8,
        'unionization_rate_pct': 11.8,
        'median_household_income': 59750,
        'pct_bachelors_degree': 29.0,
        'population_millions': 11.8,
        'dominant_industries': 'Manufacturing, Healthcare, Agriculture, Logistics',
    },
])

# Print the DataFrame as a nicely formatted table.
# to_string(index=False) drops the row index and aligns columns by width.
print("State Structural Characteristics")
print("=" * 100)
print(state_characteristics.to_string(index=False))
print("=" * 100)

In [ ]:
# ---------------------------------------------------------------
# Bar chart: compare the three states across 4 key characteristics.
# Uses a 2x2 grid because the variables are on very different
# scales (percent vs dollars) and shouldn't share an axis.
# ---------------------------------------------------------------

figures_dir = os.path.join('..', 'figures')
os.makedirs(figures_dir, exist_ok=True)

sns.set_style('whitegrid')

# Fixed state order so colors and bar positions stay consistent across runs.
state_order = ['TX', 'MA', 'OH']
df_plot = state_characteristics.set_index('state').loc[state_order]

# Assign a stable color per state.
state_colors = {'TX': 'C0', 'MA': 'C1', 'OH': 'C2'}
bar_colors = [state_colors[s] for s in state_order]

# Configure the 4 panels: (column name, panel title, y-axis label).
panels = [
    ('manufacturing_share_pct', 'Manufacturing Share of GDP', 'Percent'),
    ('unionization_rate_pct',   'Unionization Rate',           'Percent of Workers'),
    ('median_household_income', 'Median Household Income',     'US Dollars'),
    ('pct_bachelors_degree',    "Bachelor's Degree or Higher", 'Percent of Adults 25+'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Flatten the 2x2 axes array so we can zip with the panels list.
for ax, (col, title, ylabel) in zip(axes.flat, panels):
    values = df_plot[col].values
    bars = ax.bar(state_order, values, color=bar_colors, edgecolor='black', linewidth=0.6)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('State')

    # Annotate each bar with its numeric value so the chart is self-explanatory.
    # Use a thousands separator for the income panel, one decimal otherwise.
    for bar, v in zip(bars, values):
        if col == 'median_household_income':
            label = f"${v:,.0f}"
        else:
            label = f"{v:.1f}%"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha='center', va='bottom', fontsize=10,
        )

    # Add a little headroom above the tallest bar so labels don't get clipped.
    ax.set_ylim(0, max(values) * 1.15)

fig.suptitle('State Characteristics: Texas, Massachusetts, Ohio', fontsize=14, y=1.00)
plt.tight_layout()

output_path = os.path.join(figures_dir, 'state_characteristics.png')
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Saved figure to: {output_path}")

plt.show()

### Why These Differences Matter for the Phillips Curve

These structural differences set up clear, testable predictions about how the Phillips Curve should look in each state. **Ohio's** outsized manufacturing share (≈16.5% of GDP) ties its price level to globally traded goods — input costs and import competition, not local labor-market slack, do much of the work on Ohio inflation, which should flatten its observed Phillips Curve. **Texas's** heavy energy share (≈12% of GDP) means oil-price shocks move employment and prices *together* through the same channel; the resulting positive co-movement biases the unemployment–inflation relationship toward zero (or even the wrong sign) unless oil is explicitly controlled for. **Massachusetts**, by contrast, combines the region's highest unionization rate, highest median income, and a workforce in which ~45% hold a bachelor's degree — a tight, sticky-wage, high-skill labor market that the textbook theory expects to produce the **steepest** Phillips Curve of the three.

## SQL Database Construction

Holding the collected data in pandas DataFrames inside a notebook is fine for one-off exploration, but it is fragile — every analysis session has to re-pull from FRED, and any join between unemployment, inflation, controls, and characteristics has to be re-derived in code. To create a stable, queryable foundation for the rest of the project, this section loads everything into a local **SQLite database** at `../data/phillips_curve.db`.

The schema is intentionally **normalized** — five separate tables, each describing one logical entity, with `(date, state)` or `state` as the natural primary key:

| Table | Grain | Key | Source |
| --- | --- | --- | --- |
| `state_unemployment`     | state × month     | (date, state) | `unemployment_df` |
| `state_inflation`        | state × month     | (date, state) | `inflation_df` |
| `national_controls`      | month             | (date)        | `national_controls_df` |
| `state_hpi`              | state × quarter   | (date, state) | `hpi_df` |
| `state_characteristics`  | state             | (state)       | `state_characteristics` |

State codes are standardized to two-letter abbreviations (`TX`, `MA`, `OH`) across every table so that subsequent notebooks can `JOIN` cleanly on `state`. Dates are stored as ISO-format `TEXT` (`YYYY-MM-DD`), which is SQLite's recommended date storage and lexicographically sortable.

In [ ]:
# ---------------------------------------------------------------
# Build the SQLite database and populate the five tables.
# ---------------------------------------------------------------

# Ensure the data/ directory exists before opening the DB file.
db_path = os.path.join('..', 'data', 'phillips_curve.db')
os.makedirs(os.path.dirname(db_path), exist_ok=True)

# Start from a clean DB so this cell is fully idempotent: deleting the
# file before connecting guarantees we don't append to an old schema.
if os.path.exists(db_path):
    os.remove(db_path)

# sqlite3.connect() creates the file if it doesn't exist and returns a
# Connection object. cur is the Cursor we use to issue DDL statements.
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# ---------- Table 1: state_unemployment ----------
# Composite primary key (date, state) — no state can have two
# unemployment values for the same month.
cur.execute("""
    CREATE TABLE state_unemployment (
        date              TEXT NOT NULL,
        state             TEXT NOT NULL,
        unemployment_rate REAL,
        PRIMARY KEY (date, state)
    )
""")

# unemployment_df uses full state names ('Texas', 'Massachusetts', 'Ohio').
# Every other table uses two-letter codes, so we normalize here for joins.
state_name_to_code = {'Texas': 'TX', 'Massachusetts': 'MA', 'Ohio': 'OH'}
unemp_load = unemployment_df.copy()
unemp_load['state'] = unemp_load['state'].map(state_name_to_code)
# Store dates as ISO strings (SQLite's recommended TEXT date format).
unemp_load['date'] = pd.to_datetime(unemp_load['date']).dt.strftime('%Y-%m-%d')
unemp_load = unemp_load[['date', 'state', 'unemployment_rate']]
# if_exists='append' keeps the CREATE TABLE schema (with the PK) that we
# just defined; 'replace' would drop and re-create without the constraint.
unemp_load.to_sql('state_unemployment', conn, if_exists='append', index=False)

# ---------- Table 2: state_inflation ----------
cur.execute("""
    CREATE TABLE state_inflation (
        date               TEXT NOT NULL,
        state              TEXT NOT NULL,
        cpi_index          REAL,
        inflation_rate_yoy REAL,
        PRIMARY KEY (date, state)
    )
""")

infl_load = inflation_df.copy()
infl_load['date'] = pd.to_datetime(infl_load['date']).dt.strftime('%Y-%m-%d')
# Drop the cpi_source helper column; it isn't part of the schema.
infl_load = infl_load[['date', 'state', 'cpi_index', 'inflation_rate_yoy']]
infl_load.to_sql('state_inflation', conn, if_exists='append', index=False)

# ---------- Table 3: national_controls ----------
# Single primary key on date — one row per month, no state dimension.
cur.execute("""
    CREATE TABLE national_controls (
        date                    TEXT PRIMARY KEY,
        national_inflation_yoy  REAL,
        oil_price_wti           REAL,
        mortgage_rate_30yr      REAL,
        fed_funds_rate          REAL
    )
""")

# national_controls_df has date as the index — pull it back into a column.
nc_load = national_controls_df.reset_index().copy()
nc_load['date'] = pd.to_datetime(nc_load['date']).dt.strftime('%Y-%m-%d')
nc_load = nc_load[
    ['date', 'national_inflation_yoy', 'oil_price_wti',
     'mortgage_rate_30yr', 'fed_funds_rate']
]
nc_load.to_sql('national_controls', conn, if_exists='append', index=False)

# ---------- Table 4: state_hpi ----------
cur.execute("""
    CREATE TABLE state_hpi (
        date  TEXT NOT NULL,
        state TEXT NOT NULL,
        hpi   REAL,
        PRIMARY KEY (date, state)
    )
""")

hpi_load = hpi_df.copy()
hpi_load['date'] = pd.to_datetime(hpi_load['date']).dt.strftime('%Y-%m-%d')
hpi_load = hpi_load[['date', 'state', 'hpi']]
hpi_load.to_sql('state_hpi', conn, if_exists='append', index=False)

# ---------- Table 5: state_characteristics ----------
# state is the natural key — one row per state, fully cross-sectional.
cur.execute("""
    CREATE TABLE state_characteristics (
        state                   TEXT PRIMARY KEY,
        manufacturing_share_pct REAL,
        energy_share_pct        REAL,
        unionization_rate_pct   REAL,
        median_household_income REAL,
        pct_bachelors_degree    REAL,
        population_millions     REAL,
        dominant_industries     TEXT
    )
""")

state_characteristics.to_sql(
    'state_characteristics', conn, if_exists='append', index=False
)

# Commit all inserts; without commit() the DB file will be empty on close.
conn.commit()

# ---------------------------------------------------------------
# Verify: list each table and its row count.
# sqlite_master is SQLite's internal catalog table that lists every
# user-defined object. Filtering type='table' and excluding system
# tables (those starting with 'sqlite_') gives our five tables.
# ---------------------------------------------------------------
print(f"SQLite database written to: {db_path}\n")
print(f"{'Table':<28} {'Rows':>8}")
print('-' * 38)

tables = cur.execute(
    "SELECT name FROM sqlite_master "
    "WHERE type='table' AND name NOT LIKE 'sqlite_%' "
    "ORDER BY name"
).fetchall()

for (table_name,) in tables:
    n = cur.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name:<28} {n:>8}")

# Close the connection so the file is fully flushed and safe to read
# from other notebooks / DB browsers.
conn.close()

## SQL Queries — Portfolio Deliverables

With the normalized database in place, this section writes a small library of SQL queries that demonstrate the analyses the database is designed to support. Each query is **saved as a standalone `.sql` file** in `../sql/` so it can live in the GitHub repository as a self-contained portfolio artifact — anyone with the database can run them directly from `sqlite3`, DBeaver, or a notebook without needing the rest of the project code.

The notebook then **loads each `.sql` file from disk** and executes it via `pd.read_sql_query()`. Reading from the file (rather than re-typing the SQL inline) proves that each file is correct and runnable on its own.

The five queries cover progressively richer SQL features:

| # | File | SQL features demonstrated |
| --- | --- | --- |
| 1 | `build_analysis_panel.sql`        | multi-table `INNER JOIN`, composite join keys |
| 2 | `average_conditions_by_state.sql` | `GROUP BY`, aggregate functions |
| 3 | `high_inflation_months.sql`       | `WHERE` filter on derived condition, ranked ordering |
| 4 | `texas_oil_relationship.sql`      | CTE with window function (`LAG ... OVER`), filter on derived YoY |
| 5 | `phillips_curve_by_era.sql`       | `CASE`-based bucketing, grouped aggregation across categorical era and state |

A sixth file, `create_tables.sql`, holds the full DDL for the schema with column-level comments — so the database can be reconstructed from raw data without re-running the notebook.

In [ ]:
# ---------------------------------------------------------------
# Write every SQL portfolio file to ../sql/ on disk.
# Each file is a fully runnable script with header comments,
# so anyone can run them against phillips_curve.db on their own.
# ---------------------------------------------------------------

sql_dir = os.path.join('..', 'sql')
os.makedirs(sql_dir, exist_ok=True)

# ---------- DDL: create_tables.sql ----------
# Annotated CREATE TABLE statements for all five tables. Lets the
# database be reconstructed without re-running the notebook code.
create_tables_sql = """-- create_tables.sql
-- Purpose: Define the full schema for phillips_curve.db.
-- All tables use ISO date strings ('YYYY-MM-DD') and two-letter state codes
-- ('TX', 'MA', 'OH') for consistent JOINs.

-- ===========================================================
-- state_unemployment: monthly seasonally adjusted state unemployment rate
-- Source: BLS Local Area Unemployment Statistics, via FRED (TXUR/MAUR/OHUR).
-- ===========================================================
CREATE TABLE IF NOT EXISTS state_unemployment (
    date              TEXT NOT NULL,  -- 'YYYY-MM-DD', month-start
    state             TEXT NOT NULL,  -- 'TX' | 'MA' | 'OH'
    unemployment_rate REAL,           -- percent of state labor force
    PRIMARY KEY (date, state)
);

-- ===========================================================
-- state_inflation: monthly CPI level and YoY inflation rate
-- Source: BLS metro-area CPI with Census-region CPI fill for gaps.
-- ===========================================================
CREATE TABLE IF NOT EXISTS state_inflation (
    date               TEXT NOT NULL,
    state              TEXT NOT NULL,
    cpi_index          REAL,           -- price index level (not a rate)
    inflation_rate_yoy REAL,           -- year-over-year percent change
    PRIMARY KEY (date, state)
);

-- ===========================================================
-- national_controls: monthly nation-wide control variables
-- Source: FRED (CPIAUCSL, DCOILWTICO, MORTGAGE30US, FEDFUNDS).
-- ===========================================================
CREATE TABLE IF NOT EXISTS national_controls (
    date                   TEXT PRIMARY KEY,  -- 'YYYY-MM-DD', one row per month
    national_inflation_yoy REAL,              -- YoY change in national CPI (%)
    oil_price_wti          REAL,              -- WTI crude, $/barrel, monthly avg
    mortgage_rate_30yr     REAL,              -- 30-yr fixed rate, monthly avg (%)
    fed_funds_rate         REAL               -- effective fed funds rate (%)
);

-- ===========================================================
-- state_hpi: quarterly FHFA House Price Index by state
-- Source: FRED (TXSTHPI / MASTHPI / OHSTHPI).
-- ===========================================================
CREATE TABLE IF NOT EXISTS state_hpi (
    date  TEXT NOT NULL,  -- quarter-end date
    state TEXT NOT NULL,
    hpi   REAL,           -- index, 1980 Q1 = 100
    PRIMARY KEY (date, state)
);

-- ===========================================================
-- state_characteristics: time-invariant structural attributes
-- Source: U.S. Census Bureau and BLS (hand-entered).
-- ===========================================================
CREATE TABLE IF NOT EXISTS state_characteristics (
    state                   TEXT PRIMARY KEY,
    manufacturing_share_pct REAL,  -- % of state GDP
    energy_share_pct        REAL,  -- % of state GDP (mining/oil/gas)
    unionization_rate_pct   REAL,  -- % of workers in unions
    median_household_income REAL,  -- USD
    pct_bachelors_degree    REAL,  -- % of adults 25+ with BA+
    population_millions     REAL,
    dominant_industries     TEXT   -- comma-separated description
);
"""

# ---------- Query 1: build_analysis_panel.sql ----------
build_analysis_panel_sql = """-- build_analysis_panel.sql
-- Purpose: Assemble the analysis-ready monthly panel by joining
-- state_unemployment, state_inflation, and national_controls.
--
-- Join logic:
--   * state_unemployment <-> state_inflation : matched on (date, state)
--   * national_controls is national so it's joined on date alone.
-- Result grain: one row per (date, state).

SELECT
    u.date,
    u.state,
    u.unemployment_rate,
    i.cpi_index,
    i.inflation_rate_yoy,
    n.national_inflation_yoy,
    n.oil_price_wti,
    n.mortgage_rate_30yr,
    n.fed_funds_rate
FROM state_unemployment AS u
INNER JOIN state_inflation AS i
    ON u.date = i.date
   AND u.state = i.state
INNER JOIN national_controls AS n
    ON u.date = n.date
ORDER BY u.state, u.date;
"""

# ---------- Query 2: average_conditions_by_state.sql ----------
average_conditions_by_state_sql = """-- average_conditions_by_state.sql
-- Purpose: Statistical profile of each state's economic experience
-- over the full sample. One row per state.
--
-- Demonstrates: INNER JOIN on composite key, GROUP BY, aggregates
-- (COUNT, AVG, MIN, MAX) over multiple columns.

SELECT
    u.state,
    COUNT(*)                                AS n_months,
    ROUND(AVG(u.unemployment_rate), 2)      AS avg_unemployment_rate,
    ROUND(MIN(u.unemployment_rate), 2)      AS min_unemployment_rate,
    ROUND(MAX(u.unemployment_rate), 2)      AS max_unemployment_rate,
    ROUND(AVG(i.inflation_rate_yoy), 2)     AS avg_inflation_rate,
    ROUND(MIN(i.inflation_rate_yoy), 2)     AS min_inflation_rate,
    ROUND(MAX(i.inflation_rate_yoy), 2)     AS max_inflation_rate
FROM state_unemployment AS u
INNER JOIN state_inflation AS i
    ON u.date = i.date
   AND u.state = i.state
GROUP BY u.state
ORDER BY u.state;
"""

# ---------- Query 3: high_inflation_months.sql ----------
high_inflation_months_sql = """-- high_inflation_months.sql
-- Purpose: List every state-month where YoY inflation exceeded 5%,
-- alongside the corresponding state unemployment rate.
--
-- Directly relevant to the Phillips-Curve question: was unemployment
-- low (consistent with the textbook trade-off) or high (i.e. stagflation)
-- in the months when prices were rising fastest?

SELECT
    i.date,
    i.state,
    ROUND(i.inflation_rate_yoy, 2) AS inflation_rate_yoy,
    ROUND(u.unemployment_rate, 2)  AS unemployment_rate
FROM state_inflation AS i
INNER JOIN state_unemployment AS u
    ON i.date = u.date
   AND i.state = u.state
WHERE i.inflation_rate_yoy > 5.0
ORDER BY i.inflation_rate_yoy DESC;
"""

# ---------- Query 4: texas_oil_relationship.sql ----------
texas_oil_relationship_sql = """-- texas_oil_relationship.sql
-- Purpose: Examine Texas inflation and unemployment in months when WTI crude
-- oil prices moved by more than 30% YoY (in either direction).
--
-- Why this matters:
-- Texas is heavily exposed to the oil and gas sector. An oil-price shock
-- can move BOTH employment AND inflation in Texas, often in the SAME
-- direction (a price spike boosts drilling employment AND raises local
-- prices). That co-movement is the opposite of the textbook Phillips-Curve
-- trade-off and biases the estimated relationship toward zero — or even
-- the wrong sign — unless oil is explicitly controlled for in the model.

WITH oil_yoy AS (
    -- Derive the 12-month percentage change in WTI from the monthly panel.
    SELECT
        date,
        oil_price_wti,
        100.0 * (oil_price_wti - LAG(oil_price_wti, 12) OVER (ORDER BY date))
              / LAG(oil_price_wti, 12) OVER (ORDER BY date) AS oil_yoy_pct
    FROM national_controls
)
SELECT
    o.date,
    ROUND(o.oil_price_wti, 2)        AS oil_price_wti,
    ROUND(o.oil_yoy_pct, 1)          AS oil_yoy_pct_change,
    ROUND(u.unemployment_rate, 2)    AS tx_unemployment_rate,
    ROUND(i.inflation_rate_yoy, 2)   AS tx_inflation_rate_yoy
FROM oil_yoy AS o
INNER JOIN state_unemployment AS u
    ON o.date = u.date
   AND u.state = 'TX'
INNER JOIN state_inflation AS i
    ON o.date = i.date
   AND i.state = 'TX'
WHERE ABS(o.oil_yoy_pct) > 30
ORDER BY o.date;
"""

# ---------- Query 5: phillips_curve_by_era.sql ----------
phillips_curve_by_era_sql = """-- phillips_curve_by_era.sql
-- Purpose: Tag each state-month with an economic era and report the
-- average unemployment and inflation rates by (era, state).
--
-- This directly sets up the central empirical question of the project:
-- has the Phillips-Curve relationship between unemployment and inflation
-- been stable over time, or did it change between the pre-crisis years,
-- the Great Recession, the long expansion, and the COVID era?
--
-- Demonstrates: CASE-based bucketing on a date column, GROUP BY across
-- two dimensions, and a custom ORDER BY that respects chronological era.

WITH era_panel AS (
    SELECT
        u.date,
        u.state,
        u.unemployment_rate,
        i.inflation_rate_yoy,
        CASE
            WHEN CAST(SUBSTR(u.date, 1, 4) AS INTEGER) BETWEEN 2000 AND 2007
                THEN 'Pre-Crisis (2000-2007)'
            WHEN CAST(SUBSTR(u.date, 1, 4) AS INTEGER) BETWEEN 2008 AND 2009
                THEN 'Great Recession (2008-2009)'
            WHEN CAST(SUBSTR(u.date, 1, 4) AS INTEGER) BETWEEN 2010 AND 2019
                THEN 'Long Expansion (2010-2019)'
            WHEN CAST(SUBSTR(u.date, 1, 4) AS INTEGER) >= 2020
                THEN 'COVID and Aftermath (2020-)'
        END AS era
    FROM state_unemployment AS u
    INNER JOIN state_inflation AS i
        ON u.date = i.date
       AND u.state = i.state
)
SELECT
    era,
    state,
    COUNT(*)                            AS n_months,
    ROUND(AVG(unemployment_rate), 2)    AS avg_unemployment_rate,
    ROUND(AVG(inflation_rate_yoy), 2)   AS avg_inflation_rate
FROM era_panel
GROUP BY era, state
ORDER BY
    CASE era
        WHEN 'Pre-Crisis (2000-2007)'      THEN 1
        WHEN 'Great Recession (2008-2009)' THEN 2
        WHEN 'Long Expansion (2010-2019)'  THEN 3
        WHEN 'COVID and Aftermath (2020-)' THEN 4
    END,
    state;
"""

# ---------- Write all files to ../sql/ ----------
# Dict keeps things tidy and makes it trivial to add more files later.
sql_files = {
    'create_tables.sql':              create_tables_sql,
    'build_analysis_panel.sql':       build_analysis_panel_sql,
    'average_conditions_by_state.sql': average_conditions_by_state_sql,
    'high_inflation_months.sql':      high_inflation_months_sql,
    'texas_oil_relationship.sql':     texas_oil_relationship_sql,
    'phillips_curve_by_era.sql':      phillips_curve_by_era_sql,
}

for filename, content in sql_files.items():
    path = os.path.join(sql_dir, filename)
    with open(path, 'w') as f:
        f.write(content)
    print(f"Saved: {path}  ({len(content.splitlines())} lines)")


# ---------------------------------------------------------------
# Helper for the cells below: read a .sql file from disk and run it.
# Reading from disk (rather than reusing the in-memory strings) proves
# the standalone files are themselves runnable.
# ---------------------------------------------------------------
def run_sql_file(filename, head_rows=15):
    db_path = os.path.join('..', 'data', 'phillips_curve.db')
    sql_path = os.path.join('..', 'sql', filename)
    with open(sql_path, 'r') as f:
        sql_text = f.read()
    with sqlite3.connect(db_path) as conn:
        result = pd.read_sql_query(sql_text, conn)
    print(f"File:  {sql_path}")
    print(f"Shape: {result.shape}")
    print()
    if head_rows is None or len(result) <= head_rows:
        print(result.to_string(index=False))
    else:
        print(result.head(head_rows).to_string(index=False))
    return result

In [ ]:
# Query 1 — build the analysis panel (three-table JOIN).
# Result is the full state x month panel with both state series
# and the four national controls attached.
panel_df = run_sql_file('build_analysis_panel.sql', head_rows=15)

In [ ]:
# Query 2 — average / min / max unemployment and inflation by state.
# Compact statistical profile of each state over the sample period.
avg_conditions_df = run_sql_file('average_conditions_by_state.sql', head_rows=None)

In [ ]:
# Query 3 — every state-month with YoY inflation above 5%, sorted hottest-first.
# Useful for inspecting whether high-inflation periods coincided
# with low or high unemployment (textbook PC vs. stagflation).
high_inflation_df = run_sql_file('high_inflation_months.sql', head_rows=15)

In [ ]:
# Query 4 — Texas inflation/unemployment in months when WTI moved >30% YoY.
# Uses a CTE + LAG window function to derive the YoY change in oil
# before joining back to the Texas state series.
texas_oil_df = run_sql_file('texas_oil_relationship.sql', head_rows=15)

In [ ]:
# Query 5 — average unemployment and inflation by (era, state).
# Uses a CASE statement to bucket dates into Pre-Crisis, Great Recession,
# Long Expansion, and COVID & Aftermath. 12 rows total (4 eras x 3 states).
era_df = run_sql_file('phillips_curve_by_era.sql', head_rows=None)

## Save Processed Data

Everything collected and validated so far now gets persisted to disk as plain CSV — the canonical input for every downstream notebook. Two files are produced:

- **`../data/processed/phillips_curve_panel.csv`** — the analysis-ready monthly panel. Built by re-running the `build_analysis_panel.sql` query against the SQLite database, then **left-joining** the state HPI table on `(date, state)`. Because HPI is published quarterly while the rest of the panel is monthly, the join leaves nulls in the off-quarter months; those are filled by **forward-filling within each state** — the standard mixed-frequency treatment in macroeconomic work, which carries each quarterly observation forward until the next release.
- **`../data/processed/state_characteristics.csv`** — the cross-sectional state characteristics table, exported as-is for reference and qualitative interpretation.

Saving as CSV (rather than relying on the SQLite file alone) makes the dataset easy to load from any tool — R, Stata, Excel, or a plain text editor — without needing the SQL client.

In [ ]:
# ---------------------------------------------------------------
# Build the final analysis panel and save processed CSVs.
# ---------------------------------------------------------------

# Ensure the processed/ output directory exists.
processed_dir = os.path.join('..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

db_path = os.path.join('..', 'data', 'phillips_curve.db')

# ---------- 1. Re-run the panel JOIN from build_analysis_panel.sql ----------
# Reading from the .sql file (rather than hardcoding the query here) keeps
# the notebook in sync with the portfolio SQL file -- a single source of truth.
panel_sql_path = os.path.join('..', 'sql', 'build_analysis_panel.sql')
with open(panel_sql_path, 'r') as f:
    panel_sql = f.read()

# ---------- 2. Read HPI from the database ----------
# We do the (date, state) left-join in pandas because pandas' groupby+ffill
# is the cleanest way to forward-fill quarterly HPI through the off months.
with sqlite3.connect(db_path) as conn:
    panel = pd.read_sql_query(panel_sql, conn)
    hpi = pd.read_sql_query('SELECT date, state, hpi FROM state_hpi', conn)

# Convert string dates back to datetime for a clean merge & later analysis.
panel['date'] = pd.to_datetime(panel['date'])
hpi['date'] = pd.to_datetime(hpi['date'])

# ---------- 3. Left-join HPI onto the monthly panel ----------
# how='left' keeps every row of the monthly panel; non-quarter months get NaN
# for hpi, which we then forward-fill.
final_panel = panel.merge(hpi, on=['date', 'state'], how='left')

# ---------- 4. Forward-fill HPI WITHIN each state ----------
# Sort by (state, date) first so the forward-fill direction is chronological.
# groupby('state').ffill() ensures Texas values never bleed into Massachusetts.
final_panel = final_panel.sort_values(['state', 'date']).reset_index(drop=True)
final_panel['hpi'] = final_panel.groupby('state')['hpi'].ffill()

# ---------- 5. Save the two CSVs ----------
panel_csv_path = os.path.join(processed_dir, 'phillips_curve_panel.csv')
characteristics_csv_path = os.path.join(processed_dir, 'state_characteristics.csv')

# index=False because the row index is a meaningless 0..N-1 counter
# and would be re-read as a column by anyone loading the CSV later.
final_panel.to_csv(panel_csv_path, index=False)
state_characteristics.to_csv(characteristics_csv_path, index=False)

print(f"Saved panel:           {panel_csv_path}")
print(f"Saved characteristics: {characteristics_csv_path}")

# ---------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL ANALYSIS PANEL — SUMMARY")
print("=" * 70)
print(f"Total observations:   {len(final_panel):,}")
print(
    f"Date range:           {final_panel['date'].min().date()} "
    f"to {final_panel['date'].max().date()}"
)
print(
    f"Number of states:     {final_panel['state'].nunique()} "
    f"({', '.join(sorted(final_panel['state'].unique()))})"
)
print(f"Number of columns:    {len(final_panel.columns)}")
print("\nColumns:")
for col in final_panel.columns:
    print(f"  - {col}  ({final_panel[col].dtype})")

print("\nMissing values per column:")
missing = final_panel.isna().sum()
for col, n in missing.items():
    pct = 100 * n / len(final_panel)
    print(f"  {col:<28} {n:>5}   ({pct:5.1f}% of rows)")
print("=" * 70)

---

## Week 1 — Complete

This notebook has built the full data foundation for the project. In one self-contained workflow it:

- **Collected raw data** from the Federal Reserve Economic Database (FRED) for Texas, Massachusetts, and Ohio from January 2000 to the present — state unemployment rates, state-level CPI (metro with regional fill where needed), national CPI, WTI crude oil prices, the 30-year mortgage rate, the federal funds rate, and the FHFA state House Price Index. Hand-entered structural characteristics (industry mix, unionization, education, income, population) supply the cross-sectional context.
- **Constructed a SQLite database** (`../data/phillips_curve.db`) with **five normalized tables** — `state_unemployment`, `state_inflation`, `national_controls`, `state_hpi`, and `state_characteristics` — using composite primary keys on `(date, state)` and standardized two-letter state codes so every table joins cleanly.
- **Produced six portfolio SQL scripts** in `../sql/` — one DDL file (`create_tables.sql`) and five analytical queries (`build_analysis_panel.sql`, `average_conditions_by_state.sql`, `high_inflation_months.sql`, `texas_oil_relationship.sql`, `phillips_curve_by_era.sql`) — each annotated and runnable standalone.
- **Saved an analysis-ready panel** at `../data/processed/phillips_curve_panel.csv`, with quarterly HPI forward-filled to a monthly grid, plus the cross-sectional `state_characteristics.csv`. These two CSVs are the canonical inputs every subsequent notebook will read.

The next notebook, **`02_exploratory_analysis.ipynb`**, will pick up from these CSVs to build the visual case for the Phillips Curve: distribution plots, time-series overlays of unemployment against inflation per state, the raw unemployment-vs-inflation scatter plots that motivate the regression, and any qualitative anomalies (oil shocks, COVID, post-2020 inflation surge) worth flagging before formal estimation begins.